# Infra-Bench — ResNet-18 Random Features Baseline (spatial split, 3 seeds)

**Purpose.** Same-category baseline for the FM linear probes. This is
NOT a supervised end-to-end model. Instead:

- Random-init ResNet-18 (same architecture as the supervised baseline)
- **Backbone frozen after random init** — no pretraining, no fine-tuning
- Only the linear classification head is trainable
- Same training protocol as FM linear probes

## Why this baseline exists

The manuscript's headline supervised ResNet-18 baseline
(`infra_fm_resnet18_supervised_v1.ipynb`) trains a full randomly-init'd
network end-to-end on Infra-Bench. That's a *different* comparison from
the FM linear probes: LP evaluates frozen pretrained features; the
supervised baseline evaluates end-to-end supervised training.

The "same category" question — *does FM pretraining beat random features
under the linear-probe protocol?* — needs a controlled comparison where
only the feature source differs. This baseline:

| Aspect | FM Linear Probes | This baseline |
|---|---|---|
| Backbone init | Pretrained (Satlas / CROMA / Prithvi / AlphaEarth) | **Random** |
| Backbone status | Frozen | **Frozen** |
| Head | Trainable linear | Trainable linear |
| Training protocol | 25 ep, lr=1e-3, AdamW, wd=1e-4, bs=16, cap=10.0, 3 seeds | Identical |
| Spatial split | `asset_id_to_split_v1.parquet` | Same |

The only difference is the source of the frozen features. If an FM's
linear probe does not beat this baseline on a class, the pretrained
features are not measurably better than random features for that class
on this domain — a genuine negative finding worth reporting.

## Protocol details (identical to FM linear probes)

- **25 epochs, batch 16, lr=1e-3, AdamW wd=1e-4**
- **Class-weighted CE, cap=10.0** — same as FM LPs (matches
  `compute_class_weights(max_weight=10.0)`)
- **`_per_sector_v2`** — same F1 definition (macro-avg of per-class F1s
  for classes in sector, computed on full test set)
- **BEST_CKPT_BEFORE_TEST** — reload best-val ckpt before test evaluate
- **3 seeds** (314, 271, 161) — deterministic per-seed head init +
  DataLoader shuffle. Note: with a frozen random backbone, seed varies
  only the head initialization and batch order; backbone randomness is
  determined by the *first* set_seed(seed) call at model construction.
  Each seed constructs a fresh backbone → different random features per
  seed. This is intentional: it captures the joint variance from
  (random backbone features × head init).

## Differences vs. the supervised ResNet-18 baseline

| Aspect | Supervised (existing) | Random features (this notebook) |
|---|---|---|
| Backbone trainable? | Yes (all params) | **No** — frozen after random init |
| Trainable params | ~11M | **~6.7K** (linear head only) |
| Optimizer scope | `model.parameters()` | `[p for p in model.parameters() if p.requires_grad]` — head only |
| BN in training mode | Yes (trainable, updates running stats) | **No** — backbone is frozen so BN stats stay at random init |
| Story | "Best achievable with same architecture, task-specific supervision" | "What random features + LP protocol can do" |

## Outputs

- `/.../results/fm_eval_resnet18_random_features_v1/resnet18_random_features_v1_seed{314,271,161}_results.json`
- `/.../results/fm_eval_resnet18_random_features_v1/resnet18_random_features_v1_aggregate.json`
- `/.../results/fm_eval_resnet18_random_features_v1/confusion_matrix_resnet18_random_features_v1_aggregate.png`

`FM_CONFIGS` in `confusion_matrices.ipynb` can add a `'resnet_random'`
entry pointing at this subdir to reuse the confusion-matrix + training-
dynamics infrastructure. Recommended color: muted grey (signals
"reference baseline" rather than "primary comparison"). See the deferred
integration section in the task handoff for the FM_CONFIGS + FM_COLORS
snippet.

## Not a fine-tune, not end-to-end supervised

To reduce ambiguity in downstream analysis:
- `condition = 'random_features_linear_probe'` in each seed result JSON
  (distinguishes from `'linear_probe'` used by the FM notebooks and
  `'end_to_end_supervised'` used by the existing ResNet-18 baseline)
- `baseline_role_note` at aggregate top level: paste-ready sentence for
  Methods explaining the same-category comparison role


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU before training.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU available: True
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


In [2]:
%%capture
# Torch + torchvision come pre-installed on Colab. Just sklearn + pyarrow.
!pip install -q scikit-learn pyarrow


In [3]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# Only load_split_artifact is strictly required. EncoderBackbone +
# LinearClassifier live under downstream.common.models and we import
# them lazily inside the backbone cell — the curation zip should ship
# them, but if it doesn't we'll fall back to an inline definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import load_split_artifact ({e}). Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip -> /content/infrabench_repo ...
done.
Could not import load_split_artifact (No module named 'curation.utils.spatial_blocking'). Using inline fallback.


In [4]:
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_resnet18_random_features_v1'
SPLIT_ARTIFACT_PATH = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.water_works':                      'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

# Per-sector partition for v2 per-sector F1.
SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# 9-channel input: S2 bands 0..6 + S1 VV/VH at 7,8. Matches supervised baseline
# exactly so the frozen random backbone sees the same input as end-to-end supervised.
BAND_INDICES = [0, 1, 2, 3, 4, 5, 6, 7, 8]
IN_CHANNELS  = 9

PERC_LO, PERC_HI = 2.0, 98.0
IMAGE_SIZE = 224

# ---- Training (matches FM linear probe protocol exactly) ----
LP_EPOCHS  = 25
LP_BATCH   = 16
LP_LR      = 1e-3
WEIGHT_CAP = 10.0             # same 10× cap as FM linear probes
WD         = 1e-4
SEEDS      = [314, 271, 161]

# ---- Smoke check config ----
SMOKE_EPOCHS = 2
SMOKE_SEED   = 314

RUN_NAME_PREFIX = 'resnet18_random_features_v1'


def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'Output dir:       {OUTPUT_DIR}')
print(f'Split artifact:   {SPLIT_ARTIFACT_PATH}')
print(f'Input channels:   {IN_CHANNELS}  (S2 [0..6] + S1 VV/VH [7,8])')
print(f'Image size:       {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Training seeds:   {SEEDS}')
print(f'Smoke: seed {SMOKE_SEED}, {SMOKE_EPOCHS} epochs')
print(f'Protocol:         {LP_EPOCHS} epochs, batch {LP_BATCH}, lr {LP_LR}, '
      f'wd {WD}, class-weight cap {WEIGHT_CAP}')
print(f'(Head-only training — backbone frozen at random init.)')


Output dir:       /content/drive/MyDrive/infra_fm/results/fm_eval_resnet18_random_features_v1
Split artifact:   /content/drive/MyDrive/infra_fm/data/spatial_split/asset_id_to_split_v1.parquet
Input channels:   9  (S2 [0..6] + S1 VV/VH [7,8])
Image size:       224x224
Training seeds:   [314, 271, 161]
Smoke: seed 314, 2 epochs
Protocol:         25 epochs, batch 16, lr 0.001, wd 0.0001, class-weight cap 10.0
(Head-only training — backbone frozen at random init.)


In [5]:
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')


def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


  [DONE]   africa                 energy     already present (949 tiles)
  [DONE]   africa                 telecom    already present (1 tiles)
  [DONE]   africa                 transport  already present (1000 tiles)
  [DONE]   africa                 water      already present (892 tiles)
  [DONE]   asia                   energy     already present (889 tiles)
  [DONE]   asia                   telecom    already present (28 tiles)
  [DONE]   asia                   transport  already present (891 tiles)
  [DONE]   asia                   water      already present (958 tiles)
  [DONE]   australia-oceania      energy     already present (1002 tiles)
  [DONE]   australia-oceania      telecom    already present (12 tiles)
  [DONE]   australia-oceania      transport  already present (1000 tiles)
  [DONE]   australia-oceania      water      already present (1000 tiles)
  [DONE]   central-america        energy     already present (999 tiles)
  [DONE]   central-america        telecom    alread

In [6]:
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr, lo=PERC_LO, hi=PERC_HI):
    img = arr.astype(np.float32)
    low = np.percentile(img, lo); high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class NpyResNet9Dataset(Dataset):
    """Self-contained 9-band loader for a single `dataset_<region>_<sector>_v1_1k/`
    folder. Selects all 9 bands (S2 [0..6] + S1 [7,8]) in native storage order
    and applies percentile_normalize (2nd/98th percentile across the whole
    9-channel tile)."""
    def __init__(self, dataset_root,
                 band_indices=BAND_INDICES,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.band_indices = list(band_indices)
        self.allowed = set(allowed_asset_types)
        self.max_required_band = max(self.band_indices)

        manifest_path = self.dataset_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'missing manifest.json: {manifest_path}')
        with manifest_path.open() as f:
            manifest = json.load(f)
        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'

        records, dropped = [], Counter()
        for r in records_in:
            at = r.get('asset_type')
            if not at or at not in self.allowed:
                dropped['filtered_type' if at else 'no_label'] += 1; continue
            img_file = r.get('image_file')
            if not img_file:
                dropped['no_image_file'] += 1; continue
            p = images_dir / img_file
            if not p.exists():
                dropped['missing_npy'] += 1; continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1:
                    dropped['too_few_bands'] += 1; continue
            except Exception:
                dropped['load_error'] += 1; continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if dropped:
            print(f'  NpyResNet9Dataset({self.dataset_root.name}): dropped '
                  f'{sum(dropped.values())} records ({dict(dropped)})')
        if not records:
            raise RuntimeError(f'no usable records in {self.dataset_root}')
        self.records = records

    def __len__(self): return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        arr = arr[self.band_indices, :, :]              # (9, H, W)
        arr = percentile_normalize(arr)                  # -> [0, 1] across all 9
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)
        return {'image': torch.from_numpy(img),
                'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']
        img = F.interpolate(img.unsqueeze(0),
                            size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False).squeeze(0)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


source_datasets = {}
for region, sector, local in ready:
    base = NpyResNet9Dataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


  NpyResNet9Dataset(dataset_africa_water_v1_1k): dropped 129 records ({'filtered_type': 129})
  NpyResNet9Dataset(dataset_asia_water_v1_1k): dropped 246 records ({'filtered_type': 246})
  NpyResNet9Dataset(dataset_australia-oceania_water_v1_1k): dropped 14 records ({'filtered_type': 14})
  NpyResNet9Dataset(dataset_central-america_water_v1_1k): dropped 105 records ({'filtered_type': 105})
  NpyResNet9Dataset(dataset_europe_water_v1_1k): dropped 171 records ({'filtered_type': 171})
  NpyResNet9Dataset(dataset_north-america_water_v1_1k): dropped 101 records ({'filtered_type': 101})
  NpyResNet9Dataset(dataset_south-america_water_v1_1k): dropped 129 records ({'filtered_type': 129})
Built 28 cell datasets


In [7]:
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles have no split assignment (excluded).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])
print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')


Loaded split artifact: 18,750 asset_id -> split entries
  splits distribution: Counter({'train': 13087, 'val': 2851, 'test': 2812})

Global: train=12428  val=2738  test=2695


In [8]:
# Diagnostic: regenerate the v1 random stratified split inside the notebook
# for an exact-match comparison against the new spatial split. Expected
# ~47% changed — matching CROMA v2 / AlphaEarth v2 / SatlasS1 v2 / SatlasS2 v2.
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)


def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')
print('\n(Expected ~47% — matches the FM v2 notebooks.)')


Diagnostic: train/val/test transition table (old random -> new spatial)
Comparing on 17,855 tiles in both old and new splits

old \ new      train       val       test
--------------------------------------------------
train           8665      1904       1887
val             1852       396        393
test            1911       433        414

Unchanged: 9,475 (53.1%)
Changed:   8,380 (46.9%)

(Expected ~47% — matches the FM v2 notebooks.)


In [9]:
import torch.nn as nn


# Try to import EncoderBackbone from the extracted curation zip. If it isn't
# available, fall back to an inline definition that matches
# downstream/common/models.py exactly. (Same as the supervised baseline notebook.)
try:
    from downstream.common.models import EncoderBackbone, LinearClassifier
    print('Imported EncoderBackbone + LinearClassifier from downstream.common.models')
except ImportError as e:
    print(f'Could not import from downstream.common.models ({e}). Using inline fallback.')
    from torchvision.models import resnet18

    class EncoderBackbone(nn.Module):
        """Random-init ResNet-18 with adaptive first-conv for arbitrary input
        channels. Mirrors downstream/common/models.py exactly."""
        def __init__(self, backbone_name='resnet18', pretrained=False, in_channels=9):
            super().__init__()
            assert backbone_name == 'resnet18', 'inline fallback only supports resnet18'
            net = resnet18(weights=None)
            self.feature_dim = net.fc.in_features
            if in_channels != 3:
                old = net.conv1
                net.conv1 = nn.Conv2d(in_channels, old.out_channels,
                                       kernel_size=old.kernel_size,
                                       stride=old.stride,
                                       padding=old.padding, bias=False)
            net.fc = nn.Identity()
            self.encoder = net
        def forward(self, x):
            return self.encoder(x)

    class LinearClassifier(nn.Module):
        def __init__(self, in_dim, num_classes):
            super().__init__()
            self.fc = nn.Linear(in_dim, num_classes)
        def forward(self, x):
            return self.fc(x)


class ResNet18RandomFeatures(nn.Module):
    """Random-init ResNet-18 encoder (FROZEN) + trainable LinearClassifier head.

    Same-category baseline for the FM linear probes. All backbone parameters
    are frozen immediately after random init — this is the operative
    difference from `ResNet18Supervised`, which trains the full network.
    Only the linear head sees gradients during training.

    Note on seed behavior: `set_seed(seed)` is called BEFORE constructing
    the model, so each seed produces a distinct random backbone AND a
    distinct head init. The reported 3-seed std thus captures joint
    variance from (random features × head init) — the appropriate
    variance measure for a random-features baseline.
    """
    NAME = 'resnet18_random_features_v1'

    def __init__(self, num_classes=len(CLASS_NAMES), in_channels=IN_CHANNELS):
        super().__init__()
        self.encoder = EncoderBackbone('resnet18', pretrained=False,
                                       in_channels=in_channels)
        self.feature_dim = self.encoder.feature_dim
        self.head = LinearClassifier(self.feature_dim, num_classes)
        # Freeze encoder. Head remains trainable (default requires_grad=True).
        for p in self.encoder.parameters():
            p.requires_grad = False

    def forward(self, x):
        # Encoder is frozen but we still want dropout/BN in eval-consistent
        # behavior during train (BN's running stats are frozen at random-init
        # values; we treat the encoder as a fixed feature extractor).
        self.encoder.eval()
        with torch.no_grad():
            feats = self.encoder(x)
        return self.head(feats)


def BACKBONE_FACTORY():
    """Called by train_one_seed to build a fresh model per seed.
    Each call produces a DIFFERENT random backbone because set_seed(seed)
    is invoked before this call in the training loop."""
    return ResNet18RandomFeatures(num_classes=len(CLASS_NAMES),
                                   in_channels=IN_CHANNELS)


# Sanity-print: total vs trainable. Should be ~11M total, ~6.7K trainable.
_m = BACKBONE_FACTORY()
_total = sum(p.numel() for p in _m.parameters())
_trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
_frozen = _total - _trainable
print(f'ResNet18RandomFeatures built.')
print(f'  Total params:      {_total:>13,}')
print(f'  Trainable params:  {_trainable:>13,}  (linear head only)')
print(f'  Frozen params:     {_frozen:>13,}  (encoder — frozen at random init)')
assert _trainable > 0 and _trainable < _total * 0.01, (
    f'Expected head-only trainable (<1% of total); got {_trainable}/{_total}.'
)
del _m


Imported EncoderBackbone + LinearClassifier from downstream.common.models
ResNet18RandomFeatures built.
  Total params:         11,201,997
  Trainable params:          6,669  (linear head only)
  Frozen params:        11,195,328  (encoder — frozen at random init)


In [10]:
from torch.optim import AdamW
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    """Inverse-frequency weights, CAPPED at max_weight. Matches FM linear
    probe convention (10.0)."""
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """v2 per-sector F1 — matches FM LP schema exactly."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition).'
        )
    return result


def train_one_seed(seed, *, train_set, val_set, test_set,
                   num_epochs=LP_EPOCHS, lr=LP_LR, batch_size=LP_BATCH,
                   run_name_override=None):
    """Random-features linear probe. Same protocol as FM LPs:
    head-only AdamW, no LR schedule, class-weighted CE with cap."""
    set_seed(seed)
    print(f'\n--- seed {seed}  lr={lr}  epochs={num_epochs}  batch={batch_size} ---')

    model = BACKBONE_FACTORY().to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total params:     {total:,}')
    print(f'  Trainable params: {trainable:,}  (head only)')

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # Head-only AdamW — matches FM LP convention. Backbone params have
    # requires_grad=False from ResNet18RandomFeatures.__init__.
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                      lr=lr, weight_decay=WD)

    run_name = run_name_override or f'{RUN_NAME_PREFIX}_seed{seed}'
    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt      = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt     = ckpt_dir / 'checkpoint_final.pt'
    metrics_jsonl  = ckpt_dir / 'metrics.jsonl'
    metrics_jsonl.write_text('', encoding='utf-8')

    history, best_val_f1, best_epoch = [], -1.0, -1
    for epoch in range(num_epochs):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        entry = {
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        }
        history.append(entry)

        with metrics_jsonl.open('a', encoding='utf-8') as f:
            f.write(json.dumps(entry) + '\n')
            f.flush(); os.fsync(f.fileno())

        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch  = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    torch.save({'epoch': num_epochs,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history, 'best_val_f1': best_val_f1,
                'best_epoch': best_epoch}, final_ckpt)

    # ============== BEST_CKPT_BEFORE_TEST ==================================
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)
    return {
        'run_name':     run_name,
        'backbone':     'resnet18_random_features_v1',
        'condition':    'random_features_linear_probe',
        'num_epochs':   num_epochs,
        'seed':         seed,
        'lr':           lr,
        'batch_size':   batch_size,
        'best_val_f1':  best_val_f1,
        'best_epoch':   best_epoch,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1':  float(np.std(tail)),
        'history':      history,
        'test':         test,
        'tested_with':  tested_with,
    }


print('Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, '
      'head-only AdamW, matches FM LP protocol).')


Device: cuda
Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, head-only AdamW, matches FM LP protocol).


In [12]:
# ============================================================================
# SMOKE CHECK — 2 epochs on seed 314, verifies:
#   - Trainable-param count is HEAD-ONLY (~6.7K, not the ~11M of the
#     supervised baseline). If this is wrong, encoder freeze failed.
#   - Loss is finite on first step
#   - Train loss decreases over 2 epochs
#   - Extrapolated wall clock (head-only training should be fast, ~3-5 min/epoch)
# Set SMOKE_ONLY = False in the multi-seed cell (below) to launch the full run.
# ============================================================================
SMOKE_ONLY = False

print(f'Smoke check: {SMOKE_EPOCHS} epochs on seed {SMOKE_SEED} at lr={LP_LR}')
print(f'  (same protocol as FM linear probes; frozen random backbone + trainable head)')
print('=' * 76)

# Verify trainable-param count BEFORE training — cheaper than running 2 epochs
# with a bug in the freeze logic.
set_seed(SMOKE_SEED)
_m = BACKBONE_FACTORY().to(DEVICE)
_total = sum(p.numel() for p in _m.parameters())
_trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
_frozen = _total - _trainable
print(f'  Total params:      {_total:>13,}')
print(f'  Trainable params:  {_trainable:>13,}  (linear head only)')
print(f'  Frozen params:     {_frozen:>13,}  (encoder)')
# Linear head is Linear(512, 13) = 512*13 + 13 = 6,669 params
_expected_head = 512 * len(CLASS_NAMES) + len(CLASS_NAMES)
assert 500 < _trainable < 15_000, (
    f'Trainable param count {_trainable:,} looks wrong for random-features baseline: '
    f'expected ~{_expected_head:,} (linear head only). '
    f'Check that ResNet18RandomFeatures freezes the encoder.'
)
assert _frozen > 10_000_000, (
    f'Frozen param count {_frozen:,} looks too low; expected ~11M for '
    f'the ResNet-18 encoder. Check encoder was constructed.'
)
del _m

smoke_result = train_one_seed(
    SMOKE_SEED,
    train_set=train_global,
    val_set=val_global,
    test_set=test_global,
    num_epochs=SMOKE_EPOCHS,
    lr=LP_LR,
    batch_size=LP_BATCH,
    run_name_override=f'{RUN_NAME_PREFIX}_SMOKE_seed{SMOKE_SEED}',
)

hist = smoke_result['history']
train_losses = [h['train_loss'] for h in hist]
val_f1s      = [h['val_macro_f1'] for h in hist]
per_epoch_s  = [h['time_s'] for h in hist]

print('\n' + '=' * 76)
print('SMOKE RESULT SUMMARY')
print('=' * 76)
print(f'  train_loss trajectory:   {[f"{l:.4f}" for l in train_losses]}')
print(f'  val_macro_f1 trajectory: {[f"{f:.4f}" for f in val_f1s]}')
print(f'  time per epoch:          {[f"{t:.0f}s" for t in per_epoch_s]}')

def _fmt_hms(s):
    s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
    return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
avg_epoch_s = float(np.mean(per_epoch_s))
extrapolated_per_seed = avg_epoch_s * LP_EPOCHS
extrapolated_total    = extrapolated_per_seed * len(SEEDS)
print(f'\n  Extrapolated per-seed:   {_fmt_hms(extrapolated_per_seed)}  '
      f'({LP_EPOCHS} epochs × {avg_epoch_s:.0f}s/epoch)')
print(f'  Extrapolated total:      {_fmt_hms(extrapolated_total)}  '
      f'({len(SEEDS)} seeds × {_fmt_hms(extrapolated_per_seed)})')

# Divergence check
if len(train_losses) >= 2 and train_losses[-1] >= train_losses[0]:
    print()
    print('!!' + '=' * 74)
    print('!! WARNING: train_loss did NOT decrease over the smoke run.')
    print(f'!!   epoch 1 loss: {train_losses[0]:.4f}')
    print(f'!!   epoch {len(train_losses)} loss: {train_losses[-1]:.4f}  '
          f'(delta {train_losses[-1] - train_losses[0]:+.4f})')
    print('!!')
    print('!!   Random features on hard tasks may have near-flat loss curves —')
    print('!!   the head is small (6.7K params) and features are uninformative.')
    print('!!   Check val_macro_f1 trajectory too — if it also stays flat around')
    print('!!   uniform-random baseline (~1/13 = 0.077), that\'s the expected')
    print('!!   "random features can\'t learn this task" outcome.')
    print('!!' + '=' * 74)
else:
    delta = train_losses[-1] - train_losses[0]
    print(f'\n  Train loss decreased by {-delta:.4f} over smoke. Ready for full run.')

# Save smoke output
smoke_out_path = Path(OUTPUT_DIR) / 'smoke_check_results.json'
with open(smoke_out_path, 'w') as f:
    json.dump(smoke_result, f, indent=2)
print(f'\nSmoke results written: {smoke_out_path}')

print(f'\nSMOKE_ONLY = {SMOKE_ONLY} — the multi-seed cell below will '
      f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


Smoke check: 2 epochs on seed 314 at lr=0.001
  (same protocol as FM linear probes; frozen random backbone + trainable head)
  Total params:         11,201,997
  Trainable params:          6,669  (linear head only)
  Frozen params:        11,195,328  (encoder)

--- seed 314  lr=0.001  epochs=2  batch=16 ---
  Total params:     11,201,997
  Trainable params: 6,669  (head only)
  ep   1  loss=2.5444  val_acc=0.2856  val_f1=0.0679 *
  ep   2  loss=2.4922  val_acc=0.0548  val_f1=0.0165
  [BEST_CKPT_BEFORE_TEST] restored epoch 1 (val_f1=0.0679) before test

SMOKE RESULT SUMMARY
  train_loss trajectory:   ['2.5444', '2.4922']
  val_macro_f1 trajectory: ['0.0679', '0.0165']
  time per epoch:          ['30s', '30s']

  Extrapolated per-seed:   12m 31s  (25 epochs × 30s/epoch)
  Extrapolated total:      37m 34s  (3 seeds × 12m 31s)

  Train loss decreased by 0.0522 over smoke. Ready for full run.

Smoke results written: /content/drive/MyDrive/infra_fm/results/fm_eval_resnet18_random_features_v1

In [13]:
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping the 3-seed run. Set False (in the cell above) and re-run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global,
                                num_epochs=LP_EPOCHS,
                                lr=LP_LR,
                                batch_size=LP_BATCH)
        per_seed_results[seed] = result
        out_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        with open(out_path, 'w') as f:
            _json.dump({'linear_probe': result}, f, indent=2)
        print(f'\n  saved {out_path}')

    # ---- Aggregate ------------------------------------------------------
    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0)),
                'per_seed': [float(v) for v in arr]}

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {'class': CLASS_NAMES[i], 'idx': i,
         'mean_f1':  float(per_class_arr[:, i].mean()),
         'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
         'per_seed': [float(v) for v in per_class_arr[:, i]]}
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1'] for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.mean(f1s)),
            'std_macro_f1':  float(np.std(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
               for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.nanmean(f1s)),
            'std_macro_f1':  float(np.nanstd(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    agg['seeds']                      = SEEDS
    agg['split_artifact']             = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note'] = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']
    agg['lr']                         = LP_LR
    agg['condition']                  = 'random_features_linear_probe'
    agg['baseline_role_note'] = (
        'Same-category baseline for the FM linear probes: frozen randomly-'
        'initialized ResNet-18 features + trainable linear head, trained '
        'with the same protocol as the FM linear probes (25 epochs, lr=1e-3, '
        'AdamW wd=1e-4, batch=16, class-weight cap=10.0, 3 seeds, same '
        'spatial split, BEST_CKPT_BEFORE_TEST). The only difference vs. '
        'the FM linear probes is the source of the frozen features (random '
        'init here, pretrained FM there). Provides a lower bound for what '
        'feature quality FM pretraining should exceed. NOT to be confused '
        'with the supervised ResNet-18 baseline (`resnet18_supervised_v1`), '
        'which trains the full network end-to-end.'
    )

    agg_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_aggregate.json'
    with open(agg_path, 'w') as f:
        _json.dump(agg, f, indent=2)
    print(f'\nAggregate saved: {agg_path}')

    # ---- Inline confusion-matrix preview (Oranges, matches FM style) ---
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    CLASS_DISPLAY_NAMES = {
        'energy.transmission.substation': 'Transmission Substation',
        'energy.distribution.substation': 'Distribution Substation',
        'energy.distribution.other':      'Distribution (Other)',
        'energy.generation.power_plant':  'Power Plant',
        'energy.generation.solar_farm':   'Solar Farm',
        'energy.generation.wind_farm':    'Wind Farm',
        'water.wastewater.plant':         'Wastewater Plant',
        'water.water_works':              'Water Works',
        'water.storage_tank':             'Storage Tank',
        'transport.airport':              'Airport',
        'transport.train_station':        'Train Station',
        'transport.port_terminal':        'Port Terminal',
        'telecom.data_center':            'Data Center',
    }
    SHORT_NAMES = [CLASS_DISPLAY_NAMES[n] for n in CLASS_NAMES]

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    row_sums = cm_sum.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_sum.astype(np.float64), row_sums,
                         out=np.zeros_like(cm_sum, dtype=np.float64),
                         where=row_sums > 0)

    mf1 = agg['test_macro_f1']
    n_total = int(cm_sum.sum())
    fig, ax = plt.subplots(1, 1, figsize=(13, 12.5), constrained_layout=True)
    im = ax.imshow(cm_norm, cmap='Oranges', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(SHORT_NAMES, rotation=45, ha='right', fontsize=14)
    ax.set_yticklabels(SHORT_NAMES, fontsize=14)
    ax.set_xlabel('Predicted', fontsize=17)
    ax.set_ylabel('True',      fontsize=17)
    title = (f'n={n_total} across {len(SEEDS)} seeds\n'
             f'macro F1={mf1["mean"]:.3f} ± {mf1["std"]:.3f}')
    ax.set_title(title, fontsize=18)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]; count = cm_sum[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{count}\n({v:.0%})', ha='center', va='center',
                        color=color, fontsize=12)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label='Row-normalized')
    fig.suptitle('ResNet-18 (random features) — Aggregate Confusion Matrix',
                 fontsize=20, y=1.01)
    cm_path = Path(OUTPUT_DIR) / f'confusion_matrix_{RUN_NAME_PREFIX}_aggregate.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Confusion matrix (preview) saved: {cm_path}')

    # ---- Summary print ---------------------------------------------------
    print('\n' + '=' * 76)
    print(f'ResNet-18 random features — aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- '
          f'{agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- '
          f'{agg["test_accuracy"]["std"]:.4f}')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')



--- seed 314  lr=0.001  epochs=25  batch=16 ---
  Total params:     11,201,997
  Trainable params: 6,669  (head only)
  ep   1  loss=2.5444  val_acc=0.2856  val_f1=0.0679 *
  ep   2  loss=2.4922  val_acc=0.0548  val_f1=0.0165
  ep   3  loss=2.4453  val_acc=0.0603  val_f1=0.0267
  ep   4  loss=2.4380  val_acc=0.2776  val_f1=0.0808 *
  ep   5  loss=2.4489  val_acc=0.1505  val_f1=0.0694
  ep   6  loss=2.4276  val_acc=0.1618  val_f1=0.0791
  ep   7  loss=2.4041  val_acc=0.1490  val_f1=0.0673
  ep   8  loss=2.3849  val_acc=0.0862  val_f1=0.0445
  ep   9  loss=2.3906  val_acc=0.0709  val_f1=0.0306
  ep  10  loss=2.3872  val_acc=0.1450  val_f1=0.0549
  ep  11  loss=2.3600  val_acc=0.0559  val_f1=0.0215
  ep  12  loss=2.3464  val_acc=0.2706  val_f1=0.0695
  ep  13  loss=2.3574  val_acc=0.1457  val_f1=0.0563
  ep  14  loss=2.3499  val_acc=0.2655  val_f1=0.0935 *
  ep  15  loss=2.3467  val_acc=0.2820  val_f1=0.1014 *
  ep  16  loss=2.3461  val_acc=0.1183  val_f1=0.0456
  ep  17  loss=2.3436  va